# **Semantic Web & Linked Open Data with Python**
## บทที่ 07 — Python Semantic Web

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch07/ch07_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://rdflib.readthedocs.io/
*   https://github.com/RDFLib/pySHACL
*   https://pandas.pydata.org/

📖 เอกสารประกอบบทนี้: [`07-python-semantic-web/`](https://github.com/Rathachai/semantic-web/tree/main/07-python-semantic-web)

---

### Setup

In [1]:
!pip install -q rdflib pandas pyshacl

In [2]:
import pandas as pd
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, XSD

EX = Namespace('http://example.org/etl#')

## To Load Tabular Data ด้วย pandas

In [3]:
csv_text = """student_id,student_name,course_code,credits,grade
S001,Alice,CS501,4,A
S002,Bob,CS101,3,B+
S003,Carol,CS501,4,
"""
import io
df = pd.read_csv(io.StringIO(csv_text))
df

,student_id,student_name,course_code,credits,grade
0,S001,Alice,CS501,4,A
1,S002,Bob,CS101,3,B+
2,S003,Carol,CS501,4,NaN


## Transform: pandas DataFrame -> RDF Graph

แถวหนึ่งในตารางกลายเป็น triple หลายเส้นในกราฟ — ต้องจัดการ **ค่าว่าง (NaN)** เองเสมอ

In [4]:
g = Graph()
g.bind("ex", EX)

for _, row in df.iterrows():
    student = EX[row["student_id"]]
    g.add((student, RDF.type, EX.Student))
    g.add((student, EX.name, Literal(row["student_name"])))
    g.add((student, EX.enrolledIn, EX[row["course_code"]]))
    g.add((EX[row["course_code"]], EX.credits, Literal(int(row["credits"]), datatype=XSD.integer)))
    if pd.notna(row["grade"]):                      # ⚠️ ข้าม NaN แทนที่จะสร้าง triple ว่าง
        g.add((student, EX.grade, Literal(row["grade"])))

print(f"{len(g)} triples จาก {len(df)} แถว")

13 triples จาก 3 แถว


In [5]:
print(g.serialize(format="turtle"))

@prefix ex: <http://example.org/etl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:S001 a ex:Student ;
    ex:enrolledIn ex:CS501 ;
    ex:grade "A" ;
    ex:name "Alice" .

ex:S002 a ex:Student ;
    ex:enrolledIn ex:CS101 ;
    ex:grade "B+" ;
    ex:name "Bob" .

ex:S003 a ex:Student ;
    ex:enrolledIn ex:CS501 ;
    ex:name "Carol" .

ex:CS101 ex:credits 3 .

ex:CS501 ex:credits 4 .




## Validate ด้วย pySHACL

ตรวจว่านักศึกษาทุกคนต้องมี `ex:name` — ใช้ **Closed World** ต่างจาก RDF/OWL ทั่วไป

In [6]:
shapes_ttl = """
@prefix sh:  <http://www.w3.org/ns/shacl#> .
@prefix ex:  <http://example.org/etl#> .

ex:StudentShape a sh:NodeShape ;
    sh:targetClass ex:Student ;
    sh:property [
        sh:path ex:name ;
        sh:minCount 1 ;
        sh:datatype <http://www.w3.org/2001/XMLSchema#string> ;
    ] .
"""
from pyshacl import validate
conforms, results_graph, results_text = validate(g, shacl_graph=shapes_ttl,
                                                  data_graph_format="turtle",
                                                  shacl_graph_format="turtle")
print("conforms:", conforms)
print(results_text)

conforms: True
Validation Report
Conforms: True



## Enrich: รัน RDFS Reasoning

In [7]:
import owlrl
schema_ttl = """
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix ex:   <http://example.org/etl#> .
ex:Student rdfs:subClassOf ex:Person .
"""
g.parse(data=schema_ttl, format="turtle")
owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g)

res = g.query("SELECT ?s WHERE { ?s a ex:Person . }", initNs={"ex": EX})
for row in res:
    print(row.s)

http://example.org/etl#S002
http://example.org/etl#S003
http://example.org/etl#S001


## Query ผลลัพธ์สุดท้าย

In [8]:
res = g.query("""
    SELECT ?name ?grade WHERE {
      ?s ex:name ?name .
      OPTIONAL { ?s ex:grade ?grade . }
    }
""", initNs={"ex": EX})
for row in res:
    print(row.name, "-", row.grade or "(ยังไม่มีเกรด)")

Alice - A
Bob - B+
Carol - (ยังไม่มีเกรด)


## Export กลับเป็น pandas DataFrame

บางครั้งต้องแปลงกลับจาก RDF ไปเป็นตารางเพื่อวิเคราะห์ต่อด้วย pandas

In [9]:
rows = []
for row in g.query("SELECT ?name ?course WHERE { ?s ex:name ?name ; ex:enrolledIn ?course . }",
                    initNs={"ex": EX}):
    rows.append({"name": str(row.name), "course": str(row.course).split("#")[-1]})

pd.DataFrame(rows)

,name,course
0,Alice,CS501
1,Carol,CS501
2,Bob,CS101


## 🎯 สรุป

- ไปป์ไลน์ ETL เต็มรูปแบบ: **pandas → rdflib → pySHACL → owlrl → SPARQL**
- ⚠️ ต้องจัดการ **ค่าว่าง (NaN)** เองเสมอ — ห้ามสร้าง triple ที่มีค่าว่าง
- **pySHACL ใช้ Closed World** ต่างจาก RDF/OWL ที่ยึด Open World
- แปลงกลับจาก RDF ไปเป็น DataFrame ได้ด้วย SPARQL + `pd.DataFrame`

ลองเพิ่มแถวที่ไม่มี `student_name` เข้าไปใน CSV แล้วดูว่า pySHACL จับได้ไหม!

---

## จบบทที่ 07 🎉

ไปต่อบทถัดไป: **บทที่ 08 — OWL**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*